In [0]:
%pip install gspread oauth2client

In [0]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ["https://spreadsheets.google.com/feeds","https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("/Volumes/filestore/default/creds/creds.json", scope)
client = gspread.authorize(creds)

sheet = client.open("master-transaction-v2").sheet1

In [0]:
# Read all rows
data = sheet.get_all_values()
print(data)

In [0]:
import pandas as pd

# --- convert to Pandas DataFrame ---
df = pd.DataFrame(data[1:], columns=data[0])  # first row as header

# --- turn into Spark DataFrame ---
spark_df = spark.createDataFrame(df)

display(spark_df)   # shows table in Databricks

In [0]:
from pyspark.sql.functions import regexp_replace, col

# Remove $ and commas, then cast to float
df_clean = spark_df.withColumn(
    "Amount",
    regexp_replace(col("Amount"), "[$,]", "").cast("double")
)

In [0]:
from pyspark.sql.functions import to_date


df_clean = df_clean.withColumn("Date", to_date(col("Date"), "MM/dd/yyyy"))
display(df_clean)

In [0]:
df_clean.write.mode("overwrite").saveAsTable("workspace.default.financialsmaster")


